## Imports e Configuração de Diretórios

In [3]:
from __future__ import annotations

from pathlib import Path
import sys

import joblib
import nltk
from nltk.corpus import stopwords
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

# Localização robusta dos caminhos do projeto
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_CSV_PATH = PROJECT_ROOT / "data" / "raw" / "bbc-text.csv"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODELS_DIR / "bbc_multinomial_pipeline.joblib"

print(f"Raiz do Projeto:  {PROJECT_ROOT}")
print(f"Caminho do CSV:   {RAW_CSV_PATH}")
print(f"Destino Modelo:   {MODEL_PATH}")


Raiz do Projeto:  c:\Users\diego\Desktop\repositorios\bayes-bbc-prediction
Caminho do CSV:   c:\Users\diego\Desktop\repositorios\bayes-bbc-prediction\data\raw\bbc-text.csv
Destino Modelo:   c:\Users\diego\Desktop\repositorios\bayes-bbc-prediction\models\bbc_multinomial_pipeline.joblib


### Carregamento dos Dados e Estratificação Treino/Teste

In [4]:
df = pd.read_csv(RAW_CSV_PATH)

X = df["text"].astype(str)
y = df["category"].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print(f"Treino: {len(X_train)} amostras | Teste: {len(X_test)} amostras")


Treino: 1780 amostras | Teste: 445 amostras


### Construindo o Vocabulário Refinado de StopWords

In [5]:
nltk_stopwords = set(stopwords.words("english"))

ruidos_editoriais = {
    "said", "mr", "year", "people", "would", "new",
    "also", "could", "told", "just", "still", "may",
    "world", "bbc", "way", "like", "says", "us", "u"
}

numerais_e_neutros = {
    "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten",
    "first", "second", "third", "last", "next",
    "told", "added", "make", "made", "get", "take", "well", "back", "years", "time",
    "much", "many", "since", "week",
    "going", "good", "set", "say", "end", "number",
}

stopwords_refinadas = list(
    set(ENGLISH_STOP_WORDS)
    .union(nltk_stopwords)
    .union(ruidos_editoriais)
    .union(numerais_e_neutros)
)

print(f"Total de stopwords consolidadas: {len(stopwords_refinadas)} termos")


Total de stopwords consolidadas: 422 termos


### Definição e Ajuste do Pipeline Multinomial

In [6]:
pipeline = Pipeline([
    ("vectorizer", CountVectorizer(
        lowercase=True,
        stop_words=stopwords_refinadas,
        binary=False,       # Contagem inteira de ocorrências
        max_features=5000,
        min_df=2,
        max_df=0.95,
        token_pattern=r"(?u)\b[a-zA-Z]{3,}\b",
    )),
    ("classifier", MultinomialNB(
        alpha=1.0,          # Suavização de Laplace
        fit_prior=True,     # Priori aprendida empiricamente dos dados
    )),
])

# Treinamento formal do pipeline completo
pipeline.fit(X_train, y_train)
print("Pipeline treinado com sucesso!")


Pipeline treinado com sucesso!


### Inspeção dos Parâmetros Aprendidos

In [7]:
nb_model = pipeline.named_steps["classifier"]
vectorizer = pipeline.named_steps["vectorizer"]
classes = nb_model.classes_
vocab = vectorizer.get_feature_names_out()

print("=" * 70)
print(" PARÂMETROS ESTATÍSTICOS APRENDIDOS (MULTINOMIAL NAIVE BAYES)")
print("=" * 70)

# 1. Probabilidades a Priori
priors_df = pd.DataFrame({
    "Classe": classes,
    "Log-Priori": nb_model.class_log_prior_.round(4),
    "Priori P(C_k)": np.exp(nb_model.class_log_prior_).round(4),
})
print("\nProbabilidades a Priori das Classes:")
display(priors_df)

# 2. Dimensões da matriz de verossimilhança
print(f"\nFormato da Matriz de Log-Verossimilhança: {nb_model.feature_log_prob_.shape}")
print(f"({len(classes)} classes x {len(vocab)} palavras)")

# 3. Top 5 palavras com maior verossimilhança em cada classe
print("\nTop 5 Palavras de Maior Verossimilhança P(w_j | C_k):")
for idx, c in enumerate(classes):
    log_probs = nb_model.feature_log_prob_[idx]
    top_indices = log_probs.argsort()[-5:][::-1]
    top_words_with_prob = [f"{vocab[i]} ({np.exp(log_probs[i]):.4f})" for i in top_indices]
    print(f" - {c.upper():<14}: {', '.join(top_words_with_prob)}")
print("=" * 70)


 PARÂMETROS ESTATÍSTICOS APRENDIDOS (MULTINOMIAL NAIVE BAYES)

Probabilidades a Priori das Classes:


,Classe,Log-Priori,Priori P(C_k)
0,business,-1.4731,0.2292
1,entertainment,-1.7510,0.1736
2,politics,-1.6762,0.1871
3,sport,-1.4707,0.2298
4,tech,-1.7129,0.1803



Formato da Matriz de Log-Verossimilhança: (5, 5000)
(5 classes x 5000 palavras)

Top 5 Palavras de Maior Verossimilhança P(w_j | C_k):
 - BUSINESS      : market (0.0058), company (0.0058), growth (0.0053), firm (0.0049), bank (0.0047)
 - ENTERTAINMENT : film (0.0142), best (0.0106), music (0.0085), awards (0.0049), award (0.0044)
 - POLITICS      : labour (0.0100), government (0.0097), election (0.0078), blair (0.0073), party (0.0072)
 - SPORT         : england (0.0068), game (0.0064), win (0.0058), players (0.0045), team (0.0044)
 - TECH          : mobile (0.0060), technology (0.0059), games (0.0048), digital (0.0048), software (0.0047)


### Teste de Inferência Atômica com frases inéditas

In [8]:
exemplos_teste = [
    "Inflation rates jumped as central banks raised interest rates and stock markets fell.",
    "The Hollywood director won the Academy Award for best motion picture and star actor.",
    "Prime Minister addressed parliament over new tax legislation ahead of the general election.",
    "The striker scored a hat-trick in the Champions League final victory over England rivals.",
    "New smartphone features advanced artificial intelligence chip, OLED display and 5G network.",
]

print("=" * 80)
print(" TESTE DE INFERÊNCIA ATÔMICA COM NOVOS TEXTOS")
print("=" * 80)

for texto in exemplos_teste:
    pred = pipeline.predict([texto])[0]
    probas = pipeline.predict_proba([texto])[0]
    confianca = np.max(probas) * 100
    
    print(f"Texto   : {texto}")
    print(f"Predição: {pred.upper()} (Confiança: {confianca:.2f}%)")
    print("-" * 80)


 TESTE DE INFERÊNCIA ATÔMICA COM NOVOS TEXTOS
Texto   : Inflation rates jumped as central banks raised interest rates and stock markets fell.
Predição: BUSINESS (Confiança: 100.00%)
--------------------------------------------------------------------------------
Texto   : The Hollywood director won the Academy Award for best motion picture and star actor.
Predição: ENTERTAINMENT (Confiança: 100.00%)
--------------------------------------------------------------------------------
Texto   : Prime Minister addressed parliament over new tax legislation ahead of the general election.
Predição: POLITICS (Confiança: 100.00%)
--------------------------------------------------------------------------------
Texto   : The striker scored a hat-trick in the Champions League final victory over England rivals.
Predição: SPORT (Confiança: 100.00%)
--------------------------------------------------------------------------------
Texto   : New smartphone features advanced artificial intelligence chip, OL

### Serialização do Pipeline Treinado `(joblib)`

In [9]:
joblib.dump(pipeline, MODEL_PATH)
print(f"Modelo serializado com sucesso em: {MODEL_PATH}")
print(f"Tamanho do arquivo: {MODEL_PATH.stat().st_size / 1024:.2f} KB")

# Teste de carregamento para validação de integridade
loaded_pipe = joblib.load(MODEL_PATH)
teste_load = loaded_pipe.predict(["Chelsea won the premier league football cup"])
print(f"Validação de Carregamento: Predição = {teste_load[0]} (Correto: sport)")


Modelo serializado com sucesso em: c:\Users\diego\Desktop\repositorios\bayes-bbc-prediction\models\bbc_multinomial_pipeline.joblib
Tamanho do arquivo: 565.63 KB
Validação de Carregamento: Predição = sport (Correto: sport)
